# 📊 Problem Definition: Search Interest Forecasting for Consumer Demand Planning

## 📝 Project info:

#### 🏭 Industry  
- E-commerce & Marketing 

#### 🌐 Domain  
- Lifestyle & Consumer Themes  

## 📂 Dataset Information

#### 🔎 Sources
- **GDELT GKG** (Global Knowledge Graph)  
- **Google Trends**  
- **World Bank Open Data (WDI)**
- **Holidays(Python Library)**
#### All the datasets are **open source** and publicly available.  
- ✅ No privacy, licensing, or access restrictions  
- ✅ Safe for research and personal use 

## 🧩 ML Task
**Time Series Forecasting**


## 📌 Problem Statement
Brands, media agencies, and market intelligence firms need to **anticipate consumer demand 4–8 weeks in advance** for campaign planning, inventory decisions, and editorial calendars.  
Currently, they react to search spikes **after they occur**.  

This project aims to build a model that predicts **Search_Interest** for the upcoming week(s) per **country–category combination**, enabling **proactive decision-making**.


## 🔍 Why This Problem Matters
- EDA confirmed **Search_Velocity ↔ Search_Acceleration correlation = 0.721** → momentum is a measurable signal.  
- Dataset: **3.5+ years of weekly data** across **14 countries × 3 categories = 42 time series**.  
- Search interest range: **3–100**, with event-driven spikes → forecasting is challenging and valuable.  


## 📈 Expected ML Output
- Predicted **Search_Interest score** for the next **1–4 weeks** per country–category pair  
- Confidence interval for each forecast  


## 📏 Evaluation Metrics
- **MAE, RMSE** → absolute error on 0–100 scale  
- **MAPE** → percentage error, interpretable for stakeholders  
- **SMAPE** → symmetric version, handles low-interest weeks better  
- **Directional Accuracy** → did the model correctly predict *up vs. down*?  


In [32]:
# importing libraries
import pandas as pd
import numpy as np
import mysql.connector
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import pymysql
from dotenv import load_dotenv
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

### 📤 1. Data Loading (Loading Datafrom TIDB tables into pandas dataframe df)

In [33]:
# Load .env file from the current directory
load_dotenv()
# Get connection details
DB_HOST = os.getenv('DB_HOST')
DB_PORT = int(os.getenv('DB_PORT', 4000))
DB_USERNAME = os.getenv('DB_USERNAME')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

# Verify all variables are loaded
print("\n=== Connection Details ===")
print(f"Host: {DB_HOST}")
print(f"Port: {DB_PORT}")
print(f"Username: {DB_USERNAME}")
print(f"Database: {DB_DATABASE}")

if not all([DB_HOST, DB_USERNAME, DB_PASSWORD, DB_DATABASE]):
    print("\n✗ ERROR: Some environment variables are missing!")
    print("Check your .env file has all required values")
    sys.exit(1)

print("\n✓ All variables loaded!")


=== Connection Details ===
Host: gateway01.ap-northeast-1.prod.aws.tidbcloud.com
Port: 4000
Username: qgNsbp7EdsPhLkz.root
Database: GDELT&TRENDS

✓ All variables loaded!


In [34]:
# Test connection
try:
    connection = pymysql.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USERNAME,
        password=DB_PASSWORD,
        database=DB_DATABASE,
        charset='utf8mb4'
    )
    
    with connection.cursor() as cursor:
        cursor.execute("SELECT 1")
        result = cursor.fetchone()
        print("✓ Successfully connected to TiDB Cloud!")
        if connection.open:
            query = "SELECT * FROM main;"
            df = pd.read_sql(query, connection)
            print("Totla Records:",len(df))
            print("Total Columns", len(df.columns.to_list()))
            display(df.info())

    connection.close()
    
except pymysql.Error as e:
    print(f"✗ Connection Error: {e}")
    print(f"\nDebugging Info:")
    print(f"  Host: {DB_HOST}")
    print(f"  Port: {DB_PORT}")
    print(f"  User: {DB_USERNAME}")
    print(f"  Database: {DB_DATABASE}")

✓ Successfully connected to TiDB Cloud!
Totla Records: 8064
Total Columns 20
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8064 entries, 0 to 8063
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   week_start               8064 non-null   object 
 1   country_name             8064 non-null   object 
 2   category                 8064 non-null   object 
 3   search_interest          8064 non-null   float64
 4   search_velocity          8064 non-null   float64
 5   search_acceleration      8064 non-null   float64
 6   media_volume             8064 non-null   int64  
 7   tone_net_sentiment       8064 non-null   float64
 8   tone_positive_score      8064 non-null   float64
 9   tone_negative_score      8064 non-null   float64
 10  tone_polarity            8064 non-null   float64
 11  tone_activity_density    8064 non-null   float64
 12  tone_self_group_density  8064 non-null   float64
 13  s

None

In [35]:
df['week_start'] = pd.to_datetime(df['week_start'])

### 2. EDA
- In previous model already EDA part is done here focus only on model building

### 👩🏻‍💻 3. Modeling
#### 🛠️ 3.1 Pre-Requisites

#### 🔄 3.1.1 Missing Values Identification and Handling 

- missing value handling is already done there are no values in this dataset

In [36]:
df.isnull().sum()

week_start                 0
country_name               0
category                   0
search_interest            0
search_velocity            0
search_acceleration        0
media_volume               0
tone_net_sentiment         0
tone_positive_score        0
tone_negative_score        0
tone_polarity              0
tone_activity_density      0
tone_self_group_density    0
source_diversity           0
demand_to_hype_ratio       0
inflation_rate             0
internet_penetration       0
gdp_per_capita             0
is_holiday                 0
holiday_count              0
dtype: int64

### 3.1.2 **Stationarity Check (Augmented Dickey-Fuller Test)**

In [37]:
# Function to run ADF test on a given series
def is_series_stationary(series):
    clean_series = series.dropna()
    if len(clean_series) < 20:
        return False
    # p-value < 0.05 means data is stationary
    p_value = adfuller(clean_series)[1]
    return p_value < 0.05

stationary_counts = 0
non_stationary_counts = 0

# Loop through each of the 42 Country-Category groups directly
for (country, category), group in df.groupby(['country_name', 'category']):
    if is_series_stationary(group['search_interest']):
        stationary_counts += 1
    else:
        non_stationary_counts += 1

print(f"Stationarity Test Results for 'Search_Interest':")
print(f"  - Stationary Series:     {stationary_counts} / 42")
print(f"  - Non-Stationary Series: {non_stationary_counts} / 42")

Stationarity Test Results for 'Search_Interest':
  - Stationary Series:     20 / 42
  - Non-Stationary Series: 22 / 42


### ⚙️ 3.1.3 **Feature Engineering**
##### We use ['Country_Code', 'Category'] for grouping.

In [38]:
# Ensure dataframe is properly sorted before computing temporal features
df = df.sort_values(by=['country_name', 'category', 'week_start']).reset_index(drop=True)

# Lag Features (1, 2, and 4 weeks) for Search_Intrest and Media_Volume
for lag in [1, 2, 4]:
    df[f'search_interest_lag_{lag}'] = df.groupby(['country_name', 'category'])['search_interest'].shift(lag)

In [39]:
# Search_velocity and search_acceleration Lag
df['search_velocity_lag'] = df.groupby(['country_name', 'category'])['search_velocity'].shift(1)
df['search_acceleration_lag'] = df.groupby(['country_name','category'])['search_acceleration'].shift(1)

In [40]:
# Rolling Statistics (4-week windows)
df['search_interest_roll_mean_4w'] = df.groupby(['country_name', 'category'])['search_interest'].transform(lambda x: x.shift(1).rolling(4).mean())
df['search_interest_roll_std_4w'] = df.groupby(['country_name', 'category'])['search_interest'].transform(lambda x: x.shift(1).rolling(4).std())
df['media_volume_roll_mean_4w'] = df.groupby(['country_name', 'category'])['media_volume'].transform(lambda x: x.shift(1).rolling(4).mean())

In [41]:
week_of_year = df['week_start'].dt.isocalendar().week
df['sin_week'] = np.sin(2 * np.pi * week_of_year / 52)
df['cos_week'] = np.cos(2 * np.pi * week_of_year / 52)

### Drop rows with NaN values created by initial 4-week lookbacks

In [42]:
null_count= (df.isnull().sum()[df.isnull().sum() > 0])
max_null_count = null_count[max(null_count.to_dict(),key=null_count.get)]

In [43]:
print("length of new data frame after removing null values will be: ",len(df) - max_null_count)

length of new data frame after removing null values will be:  7896


In [44]:
df_model = df.dropna().copy().reset_index(drop=True)

### removing columns that are not necessary for the model

In [45]:
# Removing unnnecesay columns and PREVENTING DATA LEAKAGE
unnecessary_cols = ['tone_positive_score','tone_negative_score', 'tone_polarity', 
                    'tone_activity_density','tone_self_group_density', 'source_diversity',
                    'internet_penetration','is_holiday']

# Drop contemporaneous features to prevent target leakage!
# We can't use these at time 't' to predict 't'
leakage_cols = ['search_velocity', 'search_acceleration', 'media_volume', 'tone_net_sentiment']

df_model = df_model.drop(columns=unnecessary_cols + leakage_cols)

In [46]:
# Encode Categorical Variables for LightGBM
# LightGBM handles categoricals well if converted to 'category' dtype
df_model['country_name'] = df_model['country_name'].astype('category')
df_model['category'] = df_model['category'].astype('category')

print(df_model.dtypes)

week_start                      datetime64[ns]
country_name                          category
category                              category
search_interest                        float64
demand_to_hype_ratio                   float64
inflation_rate                         float64
gdp_per_capita                         float64
holiday_count                            int64
search_interest_lag_1                  float64
search_interest_lag_2                  float64
search_interest_lag_4                  float64
search_velocity_lag                    float64
search_acceleration_lag                float64
search_interest_roll_mean_4w           float64
search_interest_roll_std_4w            float64
media_volume_roll_mean_4w              float64
sin_week                               Float64
cos_week                               Float64
dtype: object


#### 🎯 3.1.4 X & y data preparation:

In [47]:
# Ensure datetime conversion
df_model['week_start'] = pd.to_datetime(df_model['week_start'])

# Create year-month period
df_model['year_month'] = df_model['week_start'].dt.to_period('M')

# Define target and features (exclude datetime/period columns)
target = 'search_interest'
features = [col for col in df_model.columns if col not in ['week_start', 'year_month', target]]

# Train/Val/Test splits (example: last 3 months test, previous 3 val)
unique_months = df_model['year_month'].sort_values().unique()
test_months = unique_months[-3:]
val_months  = unique_months[-6:-3]
train_months = unique_months[:-6]

train_df = df_model[df_model['year_month'].isin(train_months)]
val_df   = df_model[df_model['year_month'].isin(val_months)]
test_df  = df_model[df_model['year_month'].isin(test_months)]

X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]

In [48]:
train_df.columns

Index(['week_start', 'country_name', 'category', 'search_interest',
       'demand_to_hype_ratio', 'inflation_rate', 'gdp_per_capita',
       'holiday_count', 'search_interest_lag_1', 'search_interest_lag_2',
       'search_interest_lag_4', 'search_velocity_lag',
       'search_acceleration_lag', 'search_interest_roll_mean_4w',
       'search_interest_roll_std_4w', 'media_volume_roll_mean_4w', 'sin_week',
       'cos_week', 'year_month'],
      dtype='object')

### 🤖 3.2 Model + Define, Train & Study

In [49]:
# Convert to LightGBM Dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)


Training until validation scores don't improve for 50 rounds
[50]	training's l1: 2.88988	valid_1's l1: 4.97961
[100]	training's l1: 2.29203	valid_1's l1: 4.71614
[150]	training's l1: 2.10172	valid_1's l1: 4.71646
Early stopping, best iteration is:
[115]	training's l1: 2.22784	valid_1's l1: 4.70257


### ✅ 3.3 Trained Model Predictions & Evaluations

In [ ]:
# Define helper to compute metrics and diagnose fit
from sklearn.metrics import mean_squared_error, r2_score
def evaluate_model(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
    
    train_r2 = r2_score(y_train_true, y_train_pred)
    test_r2 = r2_score(y_test_true, y_test_pred)
    
    # Fit Diagnosis Rule
    r2_gap = train_r2 - test_r2
    if train_r2 < 0.5 and test_r2 < 0.5:
        fit_status = "Underfit"
    elif r2_gap > 0.15:
        fit_status = "Overfit"
    else:
        fit_status = "Well-Fit"
        
    return {
        "Models": model_name,
        "TrainLoss (RMSE)": round(train_rmse, 2),
        "TestLoss (RMSE)": round(test_rmse, 2),
        "TrainScore (R²)": round(train_r2, 2),
        "TestScore (R²)": round(test_r2, 2),
        "Bias-Variance (Fit)": fit_status
    }

results = []

# -------------------------------------------------------------
# LightGBM Evaluation
# -------------------------------------------------------------
lgb_train_pred = model.predict(X_train, num_iteration=model.best_iteration)
lgb_test_pred = model.predict(X_test, num_iteration=model.best_iteration)

results.append(evaluate_model("LightGBM (Tuned)", y_train, lgb_train_pred, y_test, lgb_test_pred))



# -------------------------------------------------------------
#  Display Results Summary Table
# -------------------------------------------------------------
results_df = pd.DataFrame(results)

# Format and display formatted table
display(results_df.style.hide(axis="index"))

Models,TrainLoss (RMSE),TestLoss (RMSE),TrainScore (R²),TestScore (R²),Bias-Variance (Fit)
LightGBM (Tuned),3.150000,6.740000,0.970000,0.880000,Well-Fit


#### 📥 3.5 Saving model & Real Time Prediction

In [ ]:
folder_path = r"E:\\ML Project Internship\\ML-Project\\app\pickles\\"
os.makedirs(folder_path, exist_ok=True)

model_file   = os.path.join(folder_path, "lightgbm_search_Intrest_forecast_model.pkl")
joblib.dump(model, model_file)
# Extract and Save Metadata & Schema
metadata = {
    'features': features,
    'categorical_features': ['country_name', 'category'],
    'categorical_categories': {
        'country_name': df_model['country_name'].cat.categories.tolist(),
        'category': df_model['category'].cat.categories.tolist()
    },
    'target': target
}

metadata_file = os.path.join(folder_path, 'lightgbm_search_Intrest_forecast_model_metadata.pkl')
joblib.dump(metadata, metadata_file)

print(f" Saved model to: {model_file}")
print(f" Saved metadata to: {metadata_file}")

 Saved model to: E:\\ML Project Internship\\ML-Project\\app\pickles\\lightgbm_search_Intrest_forecast_model.pkl
 Saved metadata to: E:\\ML Project Internship\\ML-Project\\app\pickles\\lightgbm_search_Intrest_forecast_model_metadata.pkl


In [ ]:
import io
import os

import boto3
import joblib
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv


# ------------------------------------------------------------------
# 1. Load Environment Variables
# ------------------------------------------------------------------

load_dotenv(override=True)

R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID", "").strip()
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID", "").strip()
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY", "").strip()
R2_BUCKET_NAME = os.getenv("R2_BUCKET_NAME", "").strip().strip("/")


# Validate required credentials
required_values = {
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}

missing = [name for name, value in required_values.items() if not value]

if missing:
    raise ValueError(
        f"❌ Missing R2 environment variables: {', '.join(missing)}"
    )


# ------------------------------------------------------------------
# 2. Configure Cloudflare R2 Client
# ------------------------------------------------------------------

endpoint_url = (
    f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"
)

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
    config=Config(
        signature_version="s3v4"
    )
)


# ------------------------------------------------------------------
# 3. Upload Function
# ------------------------------------------------------------------

def upload_object_to_r2(obj, r2_key: str) -> bool:
    """
    Serialize a Python object with joblib and upload it
    to Cloudflare R2.

    Returns:
        bool: True if upload succeeds, otherwise False.
    """

    clean_key = r2_key.lstrip("/")

    try:
        # ----------------------------------------------------------
        # Serialize
        # ----------------------------------------------------------

        print(
            f"⏳ Serializing '{clean_key}'...",
            end="",
            flush=True,
        )

        buffer = io.BytesIO()
        joblib.dump(obj, buffer)

        payload_bytes = buffer.getvalue()

        size_kb = len(payload_bytes) / 1024

        print(f" ✅ Done ({size_kb:.2f} KB)")

        # ----------------------------------------------------------
        # Upload
        # ----------------------------------------------------------

        print(
            f"⏳ Uploading to R2 bucket "
            f"'{R2_BUCKET_NAME}'...",
            end="",
            flush=True,
        )

        response = s3_client.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=clean_key,
            Body=payload_bytes,
            ContentType="application/octet-stream",
        )

        status_code = (
            response
            .get("ResponseMetadata", {})
            .get("HTTPStatusCode")
        )

        if status_code == 200:
            print(" ✅ Success! (HTTP 200)")
            print(
                f"   R2 object: "
                f"{R2_BUCKET_NAME}/{clean_key}"
            )
            return True

        print(
            f" ⚠️ Unexpected HTTP response: {status_code}"
        )

        return False

    except (ClientError, BotoCoreError) as e:
        print(f"\n❌ R2 upload failed: {e}")
        return False

    except Exception as e:
        print(f"\n❌ Serialization/upload error: {e}")
        return False


# ------------------------------------------------------------------
# 4. R2 Object Keys
# ------------------------------------------------------------------

r2_model_key = (
    "pickles/"
    "lightgbm_search_interest_forecast_model.pkl"
)

r2_metadata_key = (
    "pickles/"
    "lightgbm_search_interest_forecast_model_metadata.pkl"
)


# ------------------------------------------------------------------
# 5. Execute Uploads
# ------------------------------------------------------------------

model_uploaded = upload_object_to_r2(
    model,
    r2_model_key,
)

metadata_uploaded = upload_object_to_r2(
    metadata,
    r2_metadata_key,
)


# ------------------------------------------------------------------
# 6. Final Status
# ------------------------------------------------------------------

if model_uploaded and metadata_uploaded:
    print("\n✅ Model and metadata uploaded successfully.")
else:
    print("\n❌ One or more uploads failed.")

⏳ Serializing 'pickles/lightgbm_search_interest_forecast_model.pkl'... ✅ Done (317.10 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200)
   R2 object: marketing-intelligence/pickles/lightgbm_search_interest_forecast_model.pkl
⏳ Serializing 'pickles/lightgbm_search_interest_forecast_model_metadata.pkl'... ✅ Done (0.69 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200)
   R2 object: marketing-intelligence/pickles/lightgbm_search_interest_forecast_model_metadata.pkl

✅ Model and metadata uploaded successfully.


In [ ]:
# Load Saved Artifacts
model_path = "E://ML Project Internship//ML-Project//app//pickles//lightgbm_search_Intrest_forecast_model.pkl"
metadata_path = "E://ML Project Internship//ML-Project//app//pickles//lightgbm_search_Intrest_forecast_model_metadata.pkl"

loaded_model = joblib.load(model_path)
metadata = joblib.load(metadata_path)

In [ ]:
# 2. Out-of-Sample Forecasting Function (With Missing Features Fix)
def generate_forecast(df_raw, target_country, target_category, horizon_weeks=4):
    """
    Extracts recent history for a given country-category, dynamically projects 
    lags/rolling features step-by-step into the future, and yields predicted values.
    """
    # Filter and sort data for chosen time series
    series_df = df_raw[
        (df_raw['country_name'] == target_country) & 
        (df_raw['category'] == target_category)
    ].sort_values('week_start').copy().reset_index(drop=True)

    if len(series_df) < 4:
        raise ValueError("Minimum 4 historical weeks required for feature engineering.")

    # Track historical search values
    history_dates = list(series_df['week_start'])
    history_values = list(series_df['search_interest'])
    
    forecast_dates = []
    forecast_values = []
    
    last_date = history_dates[-1]
    latest_row = series_df.iloc[-1]  # Reference row for static country/macro features

    # Recursive / Iterative forecasting loop
    for w in range(1, horizon_weeks + 1):
        next_date = last_date + pd.Timedelta(weeks=w)
        forecast_dates.append(next_date)
        
        # Combine actual history and newly predicted values
        combined_values = history_values + forecast_values
        
        # Build vector for required model features dynamically
        feat_dict = {
            'country_name': target_country,
            'category': target_category,
            'search_interest_lag_1': combined_values[-1],
            'search_interest_lag_2': combined_values[-2] if len(combined_values) >= 2 else combined_values[-1],
            'search_interest_lag_4': combined_values[-4] if len(combined_values) >= 4 else combined_values[-1],
            'search_interest_roll_mean_4w': np.mean(combined_values[-4:]),
            'search_interest_roll_std_4w': np.std(combined_values[-4:]),
            'sin_week': np.sin(2 * np.pi * next_date.isocalendar().week / 52),
            'cos_week': np.cos(2 * np.pi * next_date.isocalendar().week / 52)
        }

        # Dynamically pull macro/lagged features that were present during training
        for col in metadata['features']:
            if col not in feat_dict:
                if col in series_df.columns:
                    # Carry forward the most recent value from historical dataset
                    feat_dict[col] = latest_row[col]
                else:
                    # Fill with 0 as safe fallback if column isn't in raw dataset
                    feat_dict[col] = 0

        # Construct DataFrame matching model schema
        input_df = pd.DataFrame([feat_dict])
        
        # Set categorical types matching training categories
        for col in metadata['categorical_features']:
            if col in metadata['categorical_categories']:
                categories = metadata['categorical_categories'][col]
                input_df[col] = pd.Categorical(input_df[col], categories=categories)
            else:
                input_df[col] = input_df[col].astype('category')
            
        # Reorder columns to match exact feature order expected by LightGBM
        input_df = input_df[metadata['features']]
        
        # Predict & clamp output to Google Trends scale (0 - 100)
        pred = loaded_model.predict(input_df)[0]
        pred_clamped = np.clip(pred, 0, 100)
        
        forecast_values.append(pred_clamped)

    # Return structured historical and forecast DataFrames
    df_hist = pd.DataFrame({'week_start': history_dates, 'search_interest': history_values, 'Type': 'Historical'})
    df_fore = pd.DataFrame({'week_start': forecast_dates, 'search_interest': forecast_values, 'Type': 'Forecast'})
    
    return df_hist, df_fore

# %%
# 3. Generate Forecast & Test
country = "India"
category = "Fashion_Beauty"

df_hist, df_fore = generate_forecast(df, target_country=country, target_category=category, horizon_weeks=8)

print(f"--- 🔮 4-Week Forecast: {country} | {category} ---")
display(df_hist)
display(df_fore[['week_start', 'search_interest']])

--- 🔮 4-Week Forecast: India | Fashion_Beauty ---


,week_start,search_interest,Type
0,2023-01-01,53.33,Historical
1,2023-01-08,54.06,Historical
2,2023-01-15,54.43,Historical
3,2023-01-22,57.64,Historical
4,2023-01-29,58.13,Historical
...,...,...,...
186,2026-07-26,54.19,Historical
187,2026-08-02,56.03,Historical
188,2026-08-09,57.88,Historical
189,2026-08-16,57.76,Historical


,week_start,search_interest
0,2026-08-30,58.018700
1,2026-09-06,59.235993
2,2026-09-13,60.406216
3,2026-09-20,61.597688
4,2026-09-27,63.729621
5,2026-10-04,65.402837
6,2026-10-11,70.061505
7,2026-10-18,69.537493
